In [ ]:
from dotenv import load_dotenv

from langchain_teddynote import logging
from langchain_openai import OpenAIEmbeddings
from langchain_upstage import UpstageEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import ConfigurableField

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-10")

In [ ]:
loader = TextLoader("./data/appendix-keywords.txt")
documents = loader.load()

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0)
split_docs = text_splitter.split_documents(documents)

In [ ]:
embeddings = OpenAIEmbeddings()

In [ ]:
db = FAISS.from_documents(split_docs, embeddings)

# Retriever

In [ ]:
retriever = db.as_retriever()

invoke(): 관련 문서 검색에 사용. 주어진 쿼리에 대한 관련 문서 반환.

In [ ]:
docs = retriever.invoke("임베딩(Embedding)은 무엇인가요?")

In [ ]:
for doc in docs:
    print(doc.page_content)
    print("=========================================================")

In [ ]:
# search_type: MMR (검색 유형을 지정)
retriever_mmr = db.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "k": 2, 
        "fetch_k": 10, 
        "lambda_mult": 0.6
    }
)

docs_mmr = retriever_mmr.invoke("임베딩(Embedding)은 무엇인가요?")

In [ ]:
for doc_mmr in docs_mmr:
    print(doc_mmr.page_content)
    print("=========================================================")

In [ ]:
# search_type: 유사도 점수 임계값
retriever_sim = db.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={
        "score_threshold": 0.8  # 임계값 설정
    }
)

docs_sim = retriever_sim.invoke("Word2Vec 은 무엇인가요?")

In [ ]:
for doc_sim in docs_sim:
    print(doc_sim.page_content)
    print("=========================================================")

In [ ]:
# top_k 설정: 검색 결과로 반환할 문서의 수 지정
retriever_topk = db.as_retriever(
    search_kwargs={
        "k": 1
    }
)

docs_topk = retriever.invoke("임베딩(Embedding)은 무엇인가요?")

In [ ]:
for doc_topk in docs_topk:
    print(doc_topk.page_content)
    print("=========================================================")

Configurable: 동작 설정

In [ ]:
retriever_config = db.as_retriever(
    search_kwargs={"k": 1}
).configurable_fields(
    search_type=ConfigurableField(
        id="search_type",
        name="Search Type",
        description="The search type to use"
    ), 
    search_kwargs=ConfigurableField(
        id="search_kwargs",  # 검색 매개변수의 고유 식별자를 설정
        name="Search Kwargs",  # 검색 매개변수의 이름을 설정
        description="The search kwargs to use"  # 검색 매개변수에 대한 설명을 작성
    )
)

In [ ]:
config1 = {
    "configurable": {
        # 검색 설정 지정
        "search_kwargs": {
            "k": 3
        }
    }
}

In [ ]:
docs_config1 = retriever_config.invoke("임베딩(Embedding)은 무엇인가요?", config=config1)

In [ ]:
for doc_config1 in docs_config1:
    print(doc_config1.page_content)
    print("=========================================================")

In [ ]:
config2 = {
    "configurable": {
        # 검색 설정: mmr
        "search_type": "mmr", 
        "search_kwargs": {
            "k": 2, 
            "fetch_k": 10, 
            "lambda_mult": 0.6
        }
    }
}

In [ ]:
docs_config2 = retriever_config.invoke("Word2Vec 은 무엇인가요?", config=config2)

In [ ]:
for doc_config2 in docs_config2:
    print(doc_config2.page_content)
    print("=========================================================")

# Upstage의 경우

질문용(query) 임베딩 모델과 문서 저장용 임베딩 모델이 따로 있음

In [ ]:
loader = TextLoader("./data/appendix-keywords.txt")

documents = loader.load()

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0)

split_docs = text_splitter.split_documents(documents)

In [ ]:
# 문서용 임베딩
doc_embedder = UpstageEmbeddings(model="solar-embedding-1-large-passage")

db = FAISS.from_documents(split_docs, doc_embedder)

In [ ]:
# 쿼리용 임베딩
query_embedder = UpstageEmbeddings(model="solar-embedding-1-large-query")

query_vector = query_embedder.embed_query("임베딩(Embedding)은 무엇인가요?")  # 쿼리 문장을 벡터로 변환

db.similarity_search_by_vector(query_vector, k=2)  # 벡터 유사도 검색 수행